# Análise Exploratória de Dados — FP&A XPTO Campinas
**Período:** Q1 2025 (Janeiro–Março)  
**Fonte:** `dashboard/public/dashboard_data.json`  
**Auditável:** cada seção documenta o método utilizado e os parâmetros aplicados.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── Estilo global ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#f8fafc',
    'axes.facecolor':   '#ffffff',
    'axes.edgecolor':   '#e2e8f0',
    'axes.grid':        True,
    'grid.color':       '#e2e8f0',
    'grid.linewidth':   0.6,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   10,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

NAVY    = '#111d3c'
TEAL    = '#0d9488'
GREEN   = '#059669'
RED     = '#dc2626'
AMBER   = '#d97706'
PALETTE = [TEAL, NAVY, AMBER, GREEN, RED, '#4f46e5', '#0891b2', '#7c3aed']

def fmt_brl(v):
    return f'R$ {v:,.0f}'.replace(',', '.')

print('✓ Bibliotecas carregadas')

In [ ]:
# ── Carrega dados ─────────────────────────────────────────────────────────────
ROOT      = Path('..') 
DASH_FILE = ROOT / 'dashboard' / 'public' / 'dashboard_data.json'

with open(DASH_FILE, encoding='utf-8') as f:
    dash = json.load(f)

contas   = pd.DataFrame(dash['all_contas'])
monthly  = pd.DataFrame(dash['monthly'])
by_group = pd.DataFrame(dash['by_group'])
quality  = dash['quality']

print(f"Empresa  : {dash['empresa']}")
print(f"Período  : {dash['periodo']}")
print(f"Contas   : {len(contas)} registros")
print(f"Grupos   : {contas['grupo'].nunique()} grupos de custo")
contas.head(3)

---
## 1. Auditoria de Qualidade dos Dados
**Método:** verificação de completude, valores nulos, contas zeradas e reconciliação orçado vs realizado.

In [ ]:
# ── Completude ────────────────────────────────────────────────────────────────
total = len(contas)
checks = {
    'Total de contas':              total,
    'Contas com realizado nulo':    int(contas['rea_q1'].isna().sum()),
    'Contas com orçado nulo':       int(contas['orc_q1'].isna().sum()),
    'Contas com realizado = 0':     int((contas['rea_q1'].fillna(0) == 0).sum()),
    'Contas com orçado = 0':        int((contas['orc_q1'].fillna(0) == 0).sum()),
    'Gastos não orçados':           len(quality.get('unplanned_spending', [])),
    'Contas sem mapeamento DRE':    len(quality.get('missing_contas', [])),
}

df_qual = pd.DataFrame.from_dict(checks, orient='index', columns=['Qtd'])
df_qual['% do total'] = (df_qual['Qtd'] / total * 100).round(1).astype(str) + '%'
df_qual.loc['Total de contas', '% do total'] = '—'

print('=== RELATÓRIO DE QUALIDADE ===')
print(df_qual.to_string())

# Score de qualidade
issues = sum([
    checks['Contas com realizado nulo'] > 0,
    checks['Contas com orçado nulo'] > 0,
    checks['Gastos não orçados'] > 0,
    checks['Contas sem mapeamento DRE'] > 0,
])
score = round((4 - issues) / 4 * 10, 1)
print(f'\n→ Score de qualidade: {score}/10 ({issues} problema(s) identificado(s))')

In [ ]:
# ── Visualização: completude por coluna ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Auditoria de Qualidade — Q1 2025', y=1.02)

# Nulos por coluna
null_pct = (contas[['orc_q1','rea_q1','var_rs','var_pct']].isna().sum() / total * 100)
null_pct.plot(kind='bar', ax=axes[0], color=[GREEN if v == 0 else RED for v in null_pct])
axes[0].set_title('% Valores Nulos por Campo')
axes[0].set_ylabel('%')
axes[0].set_xticklabels(['Orçado Q1','Realizado Q1','Variação R$','Variação %'], rotation=0)
for bar, v in zip(axes[0].patches, null_pct):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{v:.1f}%', ha='center', fontsize=9)

# Distribuição de contas por status
status_counts = {
    'Normal':           total - checks['Contas com realizado = 0'] - checks['Gastos não orçados'],
    'Realizado = 0':    checks['Contas com realizado = 0'],
    'Não orçado':       checks['Gastos não orçados'],
}
axes[1].pie(status_counts.values(), labels=status_counts.keys(),
            colors=[GREEN, AMBER, RED], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Status das Contas')

plt.tight_layout()
plt.show()

---
## 2. Estatísticas Descritivas por Grupo de Custo
**Método:** média, mediana, desvio padrão, Q1/Q3, IQR e assimetria (skewness) calculados sobre o valor realizado Q1 de cada conta dentro do grupo.

In [ ]:
# ── Estatísticas descritivas ──────────────────────────────────────────────────
desc = (
    contas.groupby('grupo')['rea_q1']
    .agg(
        N='count',
        Total='sum',
        Média='mean',
        Mediana='median',
        Desvio='std',
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Mín='min',
        Máx='max',
    )
    .assign(
        IQR=lambda d: d['Q3'] - d['Q1'],
        CV=lambda d: (d['Desvio'] / d['Média'].abs()).round(3),
        Skewness=lambda d: contas.groupby('grupo')['rea_q1'].apply(
            lambda x: float(stats.skew(x.dropna())) if len(x.dropna()) >= 3 else None
        ),
    )
    .sort_values('Total', ascending=False)
)

# Formata para exibição
display_cols = ['N','Total','Média','Mediana','Desvio','Q1','Q3','IQR','CV','Skewness']
fmt = {c: '{:,.0f}' for c in ['Total','Média','Mediana','Desvio','Q1','Q3','IQR','Mín','Máx']}
print('=== ESTATÍSTICAS DESCRITIVAS POR GRUPO (Realizado Q1) ===')
print(desc[display_cols].round(2).to_string())

In [ ]:
# ── Boxplot por grupo ─────────────────────────────────────────────────────────
grupos_order = desc.index.tolist()
data_by_group = [contas[contas['grupo'] == g]['rea_q1'].dropna().values for g in grupos_order]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(data_by_group, patch_artist=True, notch=False,
                medianprops={'color': NAVY, 'linewidth': 2},
                flierprops={'marker': 'o', 'markerfacecolor': RED, 'markersize': 5, 'alpha': 0.6})

for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(range(1, len(grupos_order) + 1))
ax.set_xticklabels(grupos_order, rotation=25, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e3:.0f}k'))
ax.set_title('Distribuição do Realizado Q1 por Grupo de Custo  (pontos = outliers)')
ax.set_ylabel('Valor por conta (R$)')
plt.tight_layout()
plt.show()
print('Método: Box = Q1–Q3, linha = mediana, whiskers = 1.5×IQR, pontos = outliers IQR')

---
## 3. Detecção de Outliers
**Método:** dois critérios independentes aplicados simultaneamente:  
- **IQR Tukey:** outlier se `valor < Q1 − 1.5×IQR` ou `valor > Q3 + 1.5×IQR` (dentro do grupo)  
- **Z-Score global:** outlier se `|z| > 2.0` (sobre todas as contas)

In [ ]:
# ── Detecção de outliers ──────────────────────────────────────────────────────
IQR_FACTOR   = 1.5
ZSCORE_LIMIT = 2.0

mean_all = contas['rea_q1'].mean()
std_all  = contas['rea_q1'].std()

outlier_rows = []
for grupo, grp in contas.groupby('grupo'):
    vals = grp['rea_q1'].dropna()
    if len(vals) < 3:
        continue
    q1_g, q3_g = vals.quantile(0.25), vals.quantile(0.75)
    iqr_g = q3_g - q1_g
    lower, upper = q1_g - IQR_FACTOR * iqr_g, q3_g + IQR_FACTOR * iqr_g

    for _, row in grp.iterrows():
        v = row['rea_q1']
        if pd.isna(v):
            continue
        z = (v - mean_all) / std_all if std_all > 0 else 0
        iqr_flag = v < lower or v > upper
        z_flag   = abs(z) > ZSCORE_LIMIT
        if not (iqr_flag or z_flag):
            continue
        outlier_rows.append({
            'Conta':    row['cod_conta'],
            'Nome':     row['nome_conta'][:40],
            'Grupo':    grupo,
            'Valor':    v,
            'Z-Score':  round(z, 2),
            'IQR Flag': '✓' if iqr_flag else '—',
            'Z Flag':   '✓' if z_flag else '—',
            'Severity': 'ALTO' if (iqr_flag and z_flag) else 'MÉDIO',
        })

df_outliers = pd.DataFrame(outlier_rows).sort_values('Z-Score', key=abs, ascending=False)
print(f'Parâmetros: IQR fator={IQR_FACTOR}, Z-Score limite=±{ZSCORE_LIMIT}')
print(f'Outliers detectados: {len(df_outliers)} conta(s)\n')
print(df_outliers.to_string(index=False))

In [ ]:
# ── Visualização: Z-scores ────────────────────────────────────────────────────
contas_sorted = contas.dropna(subset=['rea_q1']).copy()
contas_sorted['z_score'] = (contas_sorted['rea_q1'] - mean_all) / std_all
contas_sorted = contas_sorted.sort_values('z_score')

fig, ax = plt.subplots(figsize=(14, 4))
colors = [RED if abs(z) > ZSCORE_LIMIT else TEAL for z in contas_sorted['z_score']]
ax.bar(range(len(contas_sorted)), contas_sorted['z_score'], color=colors, width=1.0, alpha=0.8)
ax.axhline( ZSCORE_LIMIT, color=RED,   linestyle='--', linewidth=1.2, label=f'+{ZSCORE_LIMIT}σ limite')
ax.axhline(-ZSCORE_LIMIT, color=RED,   linestyle='--', linewidth=1.2, label=f'−{ZSCORE_LIMIT}σ limite')
ax.axhline(0,             color=NAVY,  linestyle='-',  linewidth=0.8)
ax.set_title('Z-Score por Conta — Realizado Q1  (vermelho = fora do limite ±2σ)')
ax.set_xlabel('Contas (ordenadas por z-score)')
ax.set_ylabel('Z-Score')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Análise de Concentração (Pareto + HHI)
**Método:**  
- **Curva de Pareto:** percentual acumulado das despesas por grupo, do maior para o menor  
- **HHI (Herfindahl-Hirschman Index):** `Σ(share²)` — varia de 0 (disperso) a 1 (concentrado); > 0.25 = alta concentração

In [ ]:
# ── Concentração ──────────────────────────────────────────────────────────────
grp_tot = (
    contas.groupby('grupo')['rea_q1'].sum()
    .reset_index()
    .rename(columns={'rea_q1': 'total'})
    .sort_values('total', ascending=False)
)
total_desp = grp_tot['total'].sum()
grp_tot['pct']     = grp_tot['total'] / total_desp
grp_tot['cum_pct'] = grp_tot['pct'].cumsum()

hhi = (grp_tot['pct'] ** 2).sum()
pareto_n = int((grp_tot['cum_pct'] <= 0.80).sum()) + 1

print(f'HHI = {hhi:.4f}  →  {"Alta concentração" if hhi > 0.25 else "Moderada" if hhi > 0.15 else "Baixa concentração"}')
print(f'{pareto_n} grupo(s) representam 80% das despesas (Regra de Pareto)')
print(f'Top 3 grupos = {grp_tot["pct"].head(3).sum():.1%} do total\n')
print(grp_tot[['grupo','total','pct','cum_pct']]
      .assign(total=lambda d: d['total'].map(fmt_brl),
              pct=lambda d: d['pct'].map('{:.1%}'.format),
              cum_pct=lambda d: d['cum_pct'].map('{:.1%}'.format))
      .rename(columns={'grupo':'Grupo','total':'Total','pct':'% Share','cum_pct':'% Acumulado'})
      .to_string(index=False))

In [ ]:
# ── Gráfico de Pareto ─────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(13, 5))

bars = ax1.bar(grp_tot['grupo'], grp_tot['total'], color=PALETTE[:len(grp_tot)], alpha=0.85)
ax1.set_ylabel('Despesa Realizada (R$)', color=NAVY)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e6:.1f}M'))
ax1.set_xticklabels(grp_tot['grupo'], rotation=25, ha='right', fontsize=9)

ax2 = ax1.twinx()
ax2.plot(grp_tot['grupo'], grp_tot['cum_pct'] * 100,
         color=RED, marker='o', linewidth=2, markersize=5, label='% Acumulado')
ax2.axhline(80, color=RED, linestyle='--', linewidth=1, alpha=0.6)
ax2.set_ylabel('% Acumulado', color=RED)
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))

for bar, v in zip(bars, grp_tot['total']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_desp * 0.005,
             f'R${v/1e3:.0f}k', ha='center', fontsize=8, color=NAVY)

ax1.set_title(f'Curva de Pareto — Concentração por Grupo  (HHI = {hhi:.3f})')
plt.tight_layout()
plt.show()

---
## 5. Análise de Variação Orçado vs Realizado
**Método:** variação absoluta (R$) e percentual por grupo; identificação de grupos acima e abaixo do orçado.

In [ ]:
# ── Variação por grupo ────────────────────────────────────────────────────────
var_grp = (
    contas.groupby('grupo')[['orc_q1','rea_q1']].sum()
    .assign(
        var_rs=lambda d: d['rea_q1'] - d['orc_q1'],
        var_pct=lambda d: ((d['rea_q1'] - d['orc_q1']) / d['orc_q1'].abs() * 100).round(1),
    )
    .sort_values('var_rs')
)

print('=== VARIAÇÃO ORÇADO vs REALIZADO POR GRUPO ===')
print(var_grp.assign(
    orc_q1=lambda d: d['orc_q1'].map(fmt_brl),
    rea_q1=lambda d: d['rea_q1'].map(fmt_brl),
    var_rs=lambda d: d['var_rs'].map(lambda v: f'+{fmt_brl(v)}' if v >= 0 else fmt_brl(v)),
    var_pct=lambda d: d['var_pct'].map(lambda v: f'{v:+.1f}%'),
).rename(columns={'orc_q1':'Orçado','rea_q1':'Realizado','var_rs':'Variação R$','var_pct':'Variação %'})
.to_string())

In [ ]:
# ── Gráfico waterfall de variação ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barras: orçado vs realizado
x = np.arange(len(var_grp))
w = 0.35
axes[0].bar(x - w/2, var_grp['orc_q1'], w, label='Orçado', color=NAVY, alpha=0.7)
axes[0].bar(x + w/2, var_grp['rea_q1'], w, label='Realizado', color=TEAL, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(var_grp.index, rotation=25, ha='right', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e3:.0f}k'))
axes[0].set_title('Orçado vs Realizado por Grupo')
axes[0].legend()

# Variação %
colors_var = [GREEN if v <= 0 else RED for v in var_grp['var_pct']]
axes[1].barh(var_grp.index, var_grp['var_pct'], color=colors_var, alpha=0.85)
axes[1].axvline(0, color=NAVY, linewidth=0.8)
axes[1].set_title('Variação % vs Orçado  (verde = abaixo do orçado)')
axes[1].set_xlabel('Variação %')
for i, (idx, v) in enumerate(var_grp['var_pct'].items()):
    axes[1].text(v + (0.3 if v >= 0 else -0.3), i, f'{v:+.1f}%',
                 va='center', ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

---
## 6. Estabilidade Mensal das Contas
**Método:** Coeficiente de Variação (CV = desvio padrão / média) calculado sobre os 3 meses do trimestre.  
- CV ≤ 30% → **Estável**  
- 30% < CV ≤ 60% → **Instável**  
- CV > 60% → **Volátil**

In [ ]:
# ── Estabilidade mensal ───────────────────────────────────────────────────────
CV_UNSTABLE = 0.30
CV_VOLATILE = 0.60

monthly_cols = ['rea_jan', 'rea_fev', 'rea_mar']
available = [c for c in monthly_cols if c in contas.columns]

if available:
    stab = contas[['cod_conta','nome_conta','grupo'] + available].copy()
    stab['cv'] = stab[available].apply(
        lambda row: (
            row.std(ddof=1) / abs(row.mean())
            if len(row.dropna()) >= 2 and abs(row.mean()) > 1
            else None
        ), axis=1
    )
    stab['stability'] = stab['cv'].apply(
        lambda v: 'Volátil' if v and v > CV_VOLATILE
                  else ('Instável' if v and v > CV_UNSTABLE
                  else ('Estável' if v is not None else 'N/A'))
    )
    summary_stab = stab['stability'].value_counts()
    print('=== ESTABILIDADE MENSAL ===')
    print(summary_stab.to_string())
    print(f'\nTop contas mais voláteis:')
    print(
        stab[stab['stability'].isin(['Volátil','Instável'])]
        .sort_values('cv', ascending=False)
        .head(10)[['cod_conta','nome_conta','grupo','cv','stability']]
        .assign(cv=lambda d: d['cv'].map('{:.1%}'.format))
        .to_string(index=False)
    )
else:
    # Fallback: usar monthly aggregado
    print('Colunas mensais por conta não disponíveis — usando série mensal agregada')
    print(monthly[['mes','variacao_pct']].to_string(index=False))

In [ ]:
# ── Tendência mensal agregada ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

meses = monthly['mes'].tolist()
x = np.arange(len(meses))

axes[0].plot(x, monthly['orcado'],    marker='s', color=NAVY,  linewidth=2, label='Orçado')
axes[0].plot(x, monthly['realizado'], marker='o', color=TEAL,  linewidth=2, label='Realizado')
axes[0].fill_between(x, monthly['orcado'], monthly['realizado'],
                     alpha=0.1, color=TEAL)
axes[0].set_xticks(x)
axes[0].set_xticklabels(meses)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e3:.0f}k'))
axes[0].set_title('Evolução Mensal — Orçado vs Realizado')
axes[0].legend()

var_colors = [GREEN if v <= 0 else RED for v in monthly['variacao_pct']]
axes[1].bar(meses, monthly['variacao_pct'], color=var_colors, alpha=0.85)
axes[1].axhline(0, color=NAVY, linewidth=0.8)
axes[1].set_title('Variação % Mensal vs Orçado')
axes[1].set_ylabel('%')
for i, v in enumerate(monthly['variacao_pct']):
    axes[1].text(i, v + (0.1 if v >= 0 else -0.3), f'{v:+.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Top Desvios — Ranking de Variação
**Método:** ranking por variação absoluta (R$) e percentual. Separado em acima e abaixo do orçado.

In [ ]:
# ── Top desvios ───────────────────────────────────────────────────────────────
TOP_N = 10

acima = contas[contas['var_rs'] > 0].nlargest(TOP_N, 'var_rs')[['cod_conta','nome_conta','grupo','orc_q1','rea_q1','var_rs','var_pct']]
abaixo = contas[contas['var_rs'] < 0].nsmallest(TOP_N, 'var_rs')[['cod_conta','nome_conta','grupo','orc_q1','rea_q1','var_rs','var_pct']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(acima['nome_conta'].str[:30], acima['var_rs'], color=RED, alpha=0.85)
axes[0].set_title(f'Top {TOP_N} Contas — Acima do Orçado (estouro)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e3:.0f}k'))
axes[0].invert_yaxis()
for i, v in enumerate(acima['var_rs']):
    axes[0].text(v + acima['var_rs'].max() * 0.01, i, fmt_brl(v), va='center', fontsize=8)

axes[1].barh(abaixo['nome_conta'].str[:30], abaixo['var_rs'].abs(), color=GREEN, alpha=0.85)
axes[1].set_title(f'Top {TOP_N} Contas — Abaixo do Orçado (economia)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e3:.0f}k'))
axes[1].invert_yaxis()
for i, v in enumerate(abaixo['var_rs']):
    axes[1].text(abs(v) + abaixo['var_rs'].abs().max() * 0.01, i, fmt_brl(v), va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 8. Sumário Executivo da EDA

# ── Sumário final ─────────────────────────────────────────────────────────────
s = dash['summary']

print('=' * 65)
print('SUMÁRIO EXECUTIVO — ANÁLISE EXPLORATÓRIA Q1 2025')
print('=' * 65)
print(f"Empresa : {dash['empresa']}  ·  Período : {dash['periodo']}")

print()
print('--- DESPESAS ---')
print(f"  Orçado    : {fmt_brl(s['total_orcado'])}")
print(f"  Realizado : {fmt_brl(s['total_realizado'])}")
print(f"  Variação  : {fmt_brl(s['variacao_rs'])}  ({s['variacao_pct']:+.1f}%)")

print()
print('--- RECEITA ---')
print(f"  Bruta real. : {fmt_brl(rec_summary['receita_bruta_realizada'])}  ({variacao_pct_rec:+.1f}% vs orçado)")
print(f"  Líquida     : {fmt_brl(rec_summary['receita_liquida_estimada'])}")

print()
print('--- MARGENS (sobre Receita Líquida) ---')
for lbl, orc, rea in [('  Margem Bruta  ', mb_orc, mb_rea),
                       ('  Margem EBITDA ', me_orc, me_rea),
                       ('  Margem Operac.', mo_orc, mo_rea)]:
    if rea is not None and orc is not None:
        delta = rea - orc
        print(f"{lbl}: {rea:.1f}%  (orçado {orc:.1f}%,  Δ {delta:+.1f} p.p.)")

print()
print('--- QUALIDADE ---')
print(f"  Contas       : {total}  ·  Grupos : {contas['grupo'].nunique()}")
print(f"  Outliers     : {len(df_outliers)} conta(s)  (IQR Tukey + Z-Score ±2σ)")
print(f"  HHI despesas : {hhi:.4f} — {'Alta' if hhi > 0.25 else 'Moderada' if hhi > 0.15 else 'Baixa'} concentração por grupo")
print(f"  HHI clientes : {conc['hhi']:.0f} — {conc['hhi_classificacao']}")
print(f"  Pareto 80%   : {pareto_n} grupo(s) representam 80% das despesas")
print(f"  Não orçados  : {checks['Gastos não orçados']} conta(s)")
print(f"  Score dados  : {score}/10")
print('=' * 65)

In [ ]:
# ── Receita — dados e KPIs ────────────────────────────────────────────────────
rec         = dash['receita']
rec_summary = rec['summary']
by_convenio = pd.DataFrame(rec['by_convenio'])
by_grupo    = pd.DataFrame(rec['by_grupo'])
conc        = rec['concentracao']

variacao_rec     = rec_summary.get('variacao_rs',  0) or 0
variacao_pct_rec = rec_summary.get('variacao_pct', 0) or 0

print('=== RESUMO DE RECEITA Q1 2025 ===')
print(f"Receita bruta realizada : {fmt_brl(rec_summary['receita_bruta_realizada'])}")
print(f"Receita bruta orçada    : {fmt_brl(rec_summary['receita_bruta_orcada'])}")
print(f"Variação                : {fmt_brl(variacao_rec)} ({variacao_pct_rec:+.1f}%)")
print(f"Receita líquida est.    : {fmt_brl(rec_summary['receita_liquida_estimada'])}")
print()
print('--- CONCENTRAÇÃO DE CLIENTES ---')
print(f"HHI convênios   : {conc['hhi']:.0f} — {conc['hhi_classificacao']}")
print(f"Top 1 convênio  : {conc['top1_convenio']} ({conc['top1_pct']:.1f}% da receita)")
print(f"Top 3 convênios : {conc['top3_pct']:.1f}% da receita")
print(f"Top 5 convênios : {conc['top5_pct']:.1f}% da receita")
print(f"Total convênios : {conc['n_convenios']}")
print()
print('--- TOP CONVÊNIOS ---')
print(by_convenio[['convenio', 'total_q1', 'pct_receita']].head(10)
      .assign(total_q1=lambda d: d['total_q1'].map(fmt_brl),
              pct_receita=lambda d: d['pct_receita'].map('{:.1f}%'.format))
      .rename(columns={'convenio': 'Convênio', 'total_q1': 'Receita Q1', 'pct_receita': '% Total'})
      .to_string(index=False))

In [ ]:
# ── Receita — visualização ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top convênios
top_conv    = by_convenio.head(10).copy()
colors_conv = [PALETTE[i % len(PALETTE)] for i in range(len(top_conv))]
bars_c = axes[0].barh(top_conv['convenio'].str[:25], top_conv['total_q1'],
                      color=colors_conv, alpha=0.85)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e6:.1f}M'))
axes[0].set_title('Top 10 Convênios — Receita Q1')
axes[0].invert_yaxis()
for bar, v in zip(bars_c, top_conv['pct_receita']):
    axes[0].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                 f'{v:.1f}%', va='center', fontsize=8)

# Por grupo de produto
colors_g = [PALETTE[i % len(PALETTE)] for i in range(len(by_grupo))]
bars_g = axes[1].bar(by_grupo['grupo_produto'], by_grupo['total_q1'],
                     color=colors_g, alpha=0.85)
axes[1].set_title('Receita por Grupo de Produto')
axes[1].set_xticklabels(by_grupo['grupo_produto'], rotation=25, ha='right', fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e6:.1f}M'))
for bar, v in zip(bars_g, by_grupo['pct_receita']):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + by_grupo['total_q1'].max() * 0.01,
                 f'{v:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. Margens e EBITDA
**Método:** demonstração de resultado resumida (DRE), comparando orçado vs realizado para receita líquida, lucro bruto, EBITDA e lucro operacional. Margens calculadas sobre a Receita Líquida (ROL).

In [ ]:
# ── Margens — dados e KPIs ────────────────────────────────────────────────────
mg = dash['margens']

linhas = [
    ('receita_bruta',           'Receita Bruta'),
    ('abatimentos',             'Abatimentos'),
    ('receita_liquida',         'Receita Líquida'),
    ('custo_servicos',          'Custo dos Serv.'),
    ('lucro_bruto',             'Lucro Bruto'),
    ('despesas_operacionais',   'Desp. Operacionais'),
    ('ebitda',                  'EBITDA'),
    ('depreciacao_amortizacao', 'D&A'),
    ('lucro_operacional',       'Lucro Operacional'),
]

rows_mg = []
for k, lbl in linhas:
    v = mg.get(k, {})
    rows_mg.append({
        'Linha':     lbl,
        'Orçado':    v.get('orcado', 0) or 0,
        'Realizado': v.get('realizado', 0) or 0,
    })

df_mg = pd.DataFrame(rows_mg)
df_mg['Variação R$'] = df_mg['Realizado'] - df_mg['Orçado']

mb_orc = mg.get('margem_bruta_pct',       {}).get('orcado')
mb_rea = mg.get('margem_bruta_pct',       {}).get('realizado')
me_orc = mg.get('margem_ebitda_pct',      {}).get('orcado')
me_rea = mg.get('margem_ebitda_pct',      {}).get('realizado')
mo_orc = mg.get('margem_operacional_pct', {}).get('orcado')
mo_rea = mg.get('margem_operacional_pct', {}).get('realizado')

print('=== DRE RESUMIDA — ORÇADO vs REALIZADO ===')
print(df_mg.assign(
    Orçado=lambda d: d['Orçado'].map(fmt_brl),
    Realizado=lambda d: d['Realizado'].map(fmt_brl),
    **{'Variação R$': lambda d: d['Variação R$'].map(
        lambda v: f'+{fmt_brl(v)}' if v >= 0 else fmt_brl(v))},
).to_string(index=False))

print()
print('--- MARGENS (sobre Receita Líquida) ---')
for lbl, orc, rea in [('Margem Bruta  ', mb_orc, mb_rea),
                       ('Margem EBITDA ', me_orc, me_rea),
                       ('Margem Operac.', mo_orc, mo_rea)]:
    if rea is not None and orc is not None:
        delta = rea - orc
        print(f"  {lbl}: {rea:.1f}%  (orçado {orc:.1f}%,  Δ {delta:+.1f} p.p.)")

In [ ]:
# ── Visualização de margens ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

x = np.arange(len(df_mg))
w = 0.35
axes[0].bar(x - w/2, df_mg['Orçado'],    w, label='Orçado',    color=NAVY, alpha=0.7)
axes[0].bar(x + w/2, df_mg['Realizado'], w, label='Realizado', color=TEAL, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_mg['Linha'], rotation=30, ha='right', fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v/1e6:.1f}M'))
axes[0].set_title('DRE Resumida — Orçado vs Realizado')
axes[0].legend()

margem_labels = ['Margem Bruta', 'Margem EBITDA', 'Margem Op.']
margem_orc    = [mb_orc or 0,   me_orc or 0,     mo_orc or 0]
margem_rea    = [mb_rea or 0,   me_rea or 0,     mo_rea or 0]
x2 = np.arange(len(margem_labels))
axes[1].bar(x2 - w/2, margem_orc, w, label='Orçado',    color=NAVY, alpha=0.7)
axes[1].bar(x2 + w/2, margem_rea, w, label='Realizado', color=TEAL, alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(margem_labels)
axes[1].set_ylabel('%')
axes[1].set_title('Margens % — sobre Receita Líquida')
axes[1].legend()
for i, (o, r) in enumerate(zip(margem_orc, margem_rea)):
    if o: axes[1].text(i - w/2, o + 0.3, f'{o:.1f}%', ha='center', fontsize=8)
    if r: axes[1].text(i + w/2, r + 0.3, f'{r:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Sumário final ─────────────────────────────────────────────────────────────
s = dash['summary']

print('=' * 60)
print('SUMÁRIO EXECUTIVO — ANÁLISE EXPLORATÓRIA Q1 2025')
print('=' * 60)
print(f"Empresa          : {dash['empresa']}")
print(f"Período          : {dash['periodo']}")
print()
print(f"Total orçado     : {fmt_brl(s['total_orcado'])}")
print(f"Total realizado  : {fmt_brl(s['total_realizado'])}")
print(f"Variação         : {fmt_brl(s['variacao_rs'])}  ({s['variacao_pct']:+.1f}%)")
print()
print(f"Contas analisadas: {total}")
print(f"Grupos de custo  : {contas['grupo'].nunique()}")
print(f"Outliers (IQR/Z) : {len(df_outliers)} conta(s)")
print(f"HHI concentração : {hhi:.4f} — {'Alta' if hhi > 0.25 else 'Moderada' if hhi > 0.15 else 'Baixa'}")
print(f"Pareto 80%       : {pareto_n} grupo(s) representam 80% das despesas")
print(f"Gastos s/ orçam. : {checks['Gastos não orçados']} conta(s)")
print(f"Qualidade dados  : {score}/10")
print('=' * 60)